**Problem Statement :**
 Hospital Patient Data Analysis Context: A hospital maintains patient records including admission details, department, diagnosis, doctor, and bill amount. You have two datasets: one with patient info and another with billing details. Some patients have blank bill amounts, and there are multiple rows for the same patient due to follow-ups.

In [ ]:
import pandas as pd

Step 1: Load patient dataset & show summary

In [ ]:
patients = pd.read_csv("/content/Patient_Data.csv")
billing = pd.read_csv("/content/Billing_Data.csv")

# Summary of patient dataset
patients.info()

# New Section

Step 2: Select only billing-relevant columns

In [ ]:
patients = patients[['PatientID', 'Department', 'Doctor', 'BillAmount']]

Step 3: Drop administrative columns

In [ ]:
patients = patients.drop(columns=['ReceptionistID', 'CheckInTime'], errors='ignore')

Step 4: Total bill amount per department (groupby)

In [ ]:
dept_bill = patients.groupby('Department')['BillAmount'].sum().reset_index()
print(dept_bill)

Step 5: Remove duplicate patient records

In [ ]:
patients = patients.drop_duplicates(subset='PatientID')

Step 6: Fill Missing BillAmount with Mean

In [ ]:
mean_bill = patients['BillAmount'].mean()
patients['BillAmount'] = patients['BillAmount'].fillna(mean_bill)

Step 7: Merge Billing Dataset with Patient Dataset

In [ ]:
merged_data = pd.merge(patients, billing, on='PatientID', how='inner')

Step 8: Concatenate new patient DataFrame (row-wise)

In [ ]:
new_patients = pd.DataFrame({
    'PatientID': [901, 902],
    'Department': ['Cardiology', 'Neurology'],
    'Doctor': ['Dr. Mehta', 'Dr. Rao'],
    'BillAmount': [12000, 15000]
})

merged_data = pd.concat([merged_data, new_patients], axis=0, ignore_index=True)

Step 9: Add New Billing Category Columns (Column-wise)

In [24]:
new_billing_cols = pd.DataFrame({
    'InsuranceCovered': ['Yes'] * len(merged_data),
    'FinalAmount': merged_data['BillAmount'] * 0.9})


In [ ]:
merged_data.info()
merged_data.head()

Department-wise Revenue

In [22]:
merged_data.groupby('Department')['FinalAmount'].sum()

,FinalAmount
Department,
Cardiology,6200.0
Dermatology,4000.0
Neurology,3500.0
Orthopedics,5000.0


Doctor Performance

In [23]:
merged_data.groupby('Doctor')['FinalAmount'].sum().sort_values(ascending=False)

,FinalAmount
Doctor,
Dr. Smith,6200.0
Dr. Lee,5000.0
Dr. Rose,4000.0
Dr. John,3500.0
Dr. Rao,0.0
Dr. Mehta,0.0


Average Bill per Department

In [25]:
merged_data.groupby('Department')['FinalAmount'].mean()

,FinalAmount
Department,
Cardiology,3100.0
Dermatology,4000.0
Neurology,3500.0
Orthopedics,5000.0


In [26]:
merged_data = pd.concat([merged_data, new_billing_cols], axis=1)

# Output
print(merged_data.head())
print(merged_data.info())

   PatientID   Department     Doctor   BillAmount InsuranceCovered  \
0        101   Cardiology  Dr. Smith  5000.000000           2000.0   
1        102    Neurology   Dr. John  6233.333333           1500.0   
2        103  Orthopedics    Dr. Lee  7500.000000           2500.0   
3        104   Cardiology  Dr. Smith  6200.000000           3000.0   
4        105  Dermatology   Dr. Rose  6233.333333           1000.0   

   FinalAmount InsuranceCovered  FinalAmount  
0       3000.0              Yes       4500.0  
1       3500.0              Yes       5610.0  
2       5000.0              Yes       6750.0  
3       3200.0              Yes       5580.0  
4       4000.0              Yes       5610.0  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   PatientID         7 non-null      int64  
 1   Department        7 non-null      object 
 2   Doct